In [90]:
# Import Libraries
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import shutil

#
from typing import List
from pydantic import BaseModel, Field
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.multi_query import (
    MultiQueryRetriever,
)

In [3]:
# Load Open API key and Document Loading
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
#print(api_key)
BASE_DIR = Path.cwd()
print(BASE_DIR)
RAG_Document = BASE_DIR / "Sudheer_Travels.pdf"
loader = PyPDFLoader(str(RAG_Document))
pages = loader.load()
print(f"Total PDF pages loaded: {len(pages)}")

c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels
Total PDF pages loaded: 14


In [4]:
# Document Metadata
print(pages[0].metadata)

{'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'creationdate': '2026-08-17T05:58:54+00:00', 'moddate': '2026-08-17T05:58:54+00:00', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\sudheer_travels\\Sudheer_Travels.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}


In [5]:
# Defing the metadata based on the Page Numbers
def identify_sudheer_travels_section(paper_page: int) -> str:
    """
    Identify the major section of the Sudheer Travels Rules & Regulations
    using its printed PDF page number.
    """
    
    if paper_page == 1:
        return "cover_page"
        
    if paper_page == 2:
        return "table_of_contents"
        
    if paper_page == 3:
        return "company_profile_and_overview"
        
    if 4 <= paper_page <= 5:
        return "major_routes_and_fleet_information"
        
    if 6 <= paper_page <= 7:
        return "booking_fares_and_pricing_structure"
        
    if paper_page == 8:
        return "booking_rules_and_ticketing_guidelines"
        
    if paper_page == 9:
        return "cancellation_rules_and_slabs"
        
    if paper_page == 10:
        return "refund_rules_and_processing"
        
    if paper_page == 11:
        return "luggage_pets_and_special_assistance"
        
    if paper_page == 12:
        return "passenger_conduct_and_safety_protocols"
        
    if paper_page == 13:
        return "customer_support_and_contact_information"
        
    return "unknown"

In [6]:
#Updating the Metadata of each page
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "document_title": "Sudheer Travels Official Rules & Regulations",
            "organization": "Sudheer Travels Pvt. Ltd.",
            "year": 2026,
            "document_type": "customer_policies_and_guidelines",
            "paper_page": paper_page,
            "section": identify_sudheer_travels_section(paper_page),
            "access_level": "public_customer_facing", 
        }
    )

In [7]:
# Displaying Metadata which was updated in previous Block
for page_document in pages:
    print(page_document.metadata)

{'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'creationdate': '2026-08-17T05:58:54+00:00', 'moddate': '2026-08-17T05:58:54+00:00', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\sudheer_travels\\Sudheer_Travels.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'document_title': 'Sudheer Travels Official Rules & Regulations', 'organization': 'Sudheer Travels Pvt. Ltd.', 'year': 2026, 'document_type': 'customer_policies_and_guidelines', 'paper_page': 1, 'section': 'cover_page', 'access_level': 'public_customer_facing'}
{'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'creationdate': '2026-08-17T05:58:54+00:00', 'moddate': '2026-08-17T05:58:54+00:00', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\sudheer_travels\\Sudheer_Travels.pdf', 'total_pages': 14, 'page': 1, 'page_label': '2', 'document_title': 'Sudheer Travels Official Rules & Regulations', 'organization': 'Sudheer Travels Pvt. Ltd.', 'year': 2026, 'document_t

In [8]:
# Chunking The Documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 14
Total chunks: 18


In [9]:
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"sudheer-travels-{paper_page}-chunk-{chunk_number}"
    )

In [10]:
for chunk in chunks:
    print("Chunk ID:", chunk.metadata.get("chunk_id"))
    print("Page:", chunk.metadata.get("paper_page"))
    print("Metadata:", chunk.metadata)
    print("-" * 80)

Chunk ID: sudheer-travels-1-chunk-0
Page: 1
Metadata: {'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'creationdate': '2026-08-17T05:58:54+00:00', 'moddate': '2026-08-17T05:58:54+00:00', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\sudheer_travels\\Sudheer_Travels.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'document_title': 'Sudheer Travels Official Rules & Regulations', 'organization': 'Sudheer Travels Pvt. Ltd.', 'year': 2026, 'document_type': 'customer_policies_and_guidelines', 'paper_page': 1, 'section': 'cover_page', 'access_level': 'public_customer_facing', 'start_index': 0, 'chunk_id': 'sudheer-travels-1-chunk-0'}
--------------------------------------------------------------------------------
Chunk ID: sudheer-travels-2-chunk-1
Page: 2
Metadata: {'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'creationdate': '2026-08-17T05:58:54+00:00', 'moddate': '2026-08-17T05:58:54+00:00', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\pytho

In [11]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [12]:
# Generate the embedding vector for your query
test_vector = embeddings.embed_query(
    "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

# Print the results to verify
print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 1536
First 10 values: [0.0301513671875, -0.03143310546875, 0.06903076171875, 0.00942230224609375, -0.041534423828125, 0.0012969970703125, -0.0269317626953125, 0.0075836181640625, 0.031341552734375, 0.0010805130004882812]


In [15]:
PERSIST_DIRECTORY = BASE_DIR / "Sudheer_Travel"

# Set this to False when you want to reuse the existing index.
REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

In [16]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="Sudheer_Travels_retriever_demo",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Vector store created successfully.
Stored chunks: 18
Persisted at: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travel


In [17]:
# Same directory used during creation
PERSIST_DIRECTORY = BASE_DIR / "Sudheer_Travel"
PERSIST_DIRECTORY

WindowsPath('c:/Sudheer/Agentic_AI/workspace/python/AI_Basics/Basics_2/sudheer_travels/Sudheer_Travel')

In [18]:
# Load the existing Chroma collection
vector_store = Chroma(
    collection_name="Sudheer_Travels_retriever_demo",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

print("Existing vector store loaded successfully.")
print(f"Persist directory: {PERSIST_DIRECTORY}")

Existing vector store loaded successfully.
Persist directory: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travel


In [19]:
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

In [20]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

In [21]:
query = "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 12
SECTION: passenger_conduct_and_safety_protocols
CHUNK ID: sudheer-travels-12-chunk-15
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travels.pdf
------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in advance.
Medical Conditions: Must carry necessary medications. Crew has basic First Aid.
Pregnant Wome

In [22]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

In [23]:
query = "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 12
SECTION: passenger_conduct_and_safety_protocols
CHUNK ID: sudheer-travels-12-chunk-15
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travels.pdf
------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in advance.
Medical Conditions: Must carry necessary medications. Crew has basic First Aid.
Pregnant Wome

In [24]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 4,
        "score_threshold": 0.50,
    }
)

In [25]:
query = "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 12
SECTION: passenger_conduct_and_safety_protocols
CHUNK ID: sudheer-travels-12-chunk-15
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travels.pdf
------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in advance.
Medical Conditions: Must carry necessary medications. Crew has basic First Aid.
Pregnant Wome

In [26]:
print("=" * 90)
print("Similarity Search results:")
print("=" * 90)
for document in similarity_documents:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )
print("=" * 90)
print("mmr results:")
print("=" * 90)
for document in mmr_documents:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )
print("=" * 90)
print("similarity_score_threshold results:")
print("=" * 90)
for document in threshold_documents:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
12 passenger_conduct_and_safety_protocols sudheer-travels-12-chunk-15
8 booking_rules_and_ticketing_guidelines sudheer-travels-8-chunk-10
9 cancellation_rules_and_slabs sudheer-travels-9-chunk-11
10 refund_rules_and_processing sudheer-travels-10-chunk-13
mmr results:
12 passenger_conduct_and_safety_protocols sudheer-travels-12-chunk-15
8 booking_rules_and_ticketing_guidelines sudheer-travels-8-chunk-10
7 booking_fares_and_pricing_structure sudheer-travels-7-chunk-9
9 cancellation_rules_and_slabs sudheer-travels-9-chunk-12
similarity_score_threshold results:
12 passenger_conduct_and_safety_protocols sudheer-travels-12-chunk-15


In [27]:
#similarity_search_with_relevance_scores
query = "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=4,
)

print(scored_results)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

[(Document(id='e221f9bb-871a-41d3-83d9-0dce47422b1e', metadata={'start_index': 0, 'moddate': '2026-08-17T05:58:54+00:00', 'producer': 'Skia/PDF m97', 'total_pages': 14, 'section': 'passenger_conduct_and_safety_protocols', 'document_type': 'customer_policies_and_guidelines', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\sudheer_travels\\Sudheer_Travels.pdf', 'page': 11, 'document_title': 'Sudheer Travels Official Rules & Regulations', 'paper_page': 12, 'creator': 'Chromium', 'year': 2026, 'page_label': '12', 'creationdate': '2026-08-17T05:58:54+00:00', 'chunk_id': 'sudheer-travels-12-chunk-15', 'access_level': 'public_customer_facing', 'organization': 'Sudheer Travels Pvt. Ltd.'}, page_content='7. Luggage, Pets & Special Assistance\n7.1 Luggage Policy\nFree Allowance: 15 kg stowed luggage + one small cabin bag per passenger.\nExcess Luggage: Charged at ₹20 per kg, subject to availability.\nProhibited Items: Flammables, explosives, corrosives, contraband, we

In [28]:
metric_query = "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")
print(candidate_texts)

Candidate chunks selected: 6
['7. Luggage, Pets & Special Assistance\n7.1 Luggage Policy\nFree Allowance: 15 kg stowed luggage + one small cabin bag per passenger.\nExcess Luggage: Charged at ₹20 per kg, subject to availability.\nProhibited Items: Flammables, explosives, corrosives, contraband, weapons,\nstrong-smelling goods.\nValuables: Sudheer Travels assumes no liability for lost or stolen valuables.\n7.2 Pet Policy\nNo Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage\ncompartments.\n7.3 Passengers Requiring Special Assistance\nWheelchair Assistance: Ground staff can assist. Notify 24 hours in advance.\nMedical Conditions: Must carry necessary medications. Crew has basic First Aid.\nPregnant Women: Consult a physician if beyond 28 weeks of gestation.', 'Defense Personnel: A 15% discount is provided to active and retired defense\npersonnel upon presentation of a valid military ID.\nInfants and Children: Children below the age of 3 years travel free p

In [29]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

print(query_vector)
print(document_vectors)

Query-vector shape: (1536,)
Document-vectors shape: (6, 1536)
[ 0.03015137 -0.03143311  0.06903076 ... -0.00621796  0.03096008
  0.01828003]
[[ 4.60815430e-03  1.52816772e-02  5.38024902e-02 ...  2.64282227e-02
   1.06430054e-02 -1.11999512e-02]
 [ 2.85644531e-02  2.28424072e-02  6.16149902e-02 ...  1.93786621e-02
   1.33438110e-02 -1.80206299e-02]
 [-1.60369873e-02 -2.48908997e-03  5.14526367e-02 ...  2.39105225e-02
  -9.92774963e-04 -3.99475098e-02]
 [-4.76684570e-02  4.48303223e-02  5.04455566e-02 ...  2.02178955e-02
  -4.69970703e-03 -2.06909180e-02]
 [-2.85034180e-02 -2.70996094e-02  5.64575195e-02 ...  3.96423340e-02
  -9.38773155e-05 -1.24053955e-02]
 [ 7.68661499e-03 -2.39562988e-02  4.59289551e-02 ...  3.53698730e-02
  -5.06877899e-04 -2.95715332e-02]]


In [30]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

In [31]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,12,passenger_conduct_and_safety_protocols,sudheer-travels-12-chunk-15,0.507280,0.992625,0.507209,"7. Luggage, Pets & Special Assistance 7.1 Lugg..."
1,8,booking_rules_and_ticketing_guidelines,sudheer-travels-8-chunk-10,0.343788,1.145572,0.343765,Defense Personnel: A 15% discount is provided ...
2,9,cancellation_rules_and_slabs,sudheer-travels-9-chunk-11,0.337216,1.151215,0.337147,4. Booking Rules & Ticketing Guidelines 4.1 Ti...
3,10,refund_rules_and_processing,sudheer-travels-10-chunk-13,0.333439,1.154406,0.333322,5. Cancellation Rules & Slabs 5.1 Standard Can...
4,7,booking_fares_and_pricing_structure,sudheer-travels-7-chunk-8,0.331997,1.155828,0.331980,3. Booking Fares & Pricing Structure 3.1 Dynam...
5,7,booking_fares_and_pricing_structure,sudheer-travels-7-chunk-9,0.330912,1.156771,0.330898,"Kakinada to Mumbai/Pune Volvo AC Sleeper ₹2,50..."


In [32]:
metric_query

'What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?'

In [33]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,12,passenger_conduct_and_safety_protocols,0.507280,"7. Luggage, Pets & Special Assistance 7.1 Lugg..."
1,8,booking_rules_and_ticketing_guidelines,0.343788,Defense Personnel: A 15% discount is provided ...
2,9,cancellation_rules_and_slabs,0.337216,4. Booking Rules & Ticketing Guidelines 4.1 Ti...
3,10,refund_rules_and_processing,0.333439,5. Cancellation Rules & Slabs 5.1 Standard Can...
4,7,booking_fares_and_pricing_structure,0.331997,3. Booking Fares & Pricing Structure 3.1 Dynam...
5,7,booking_fares_and_pricing_structure,0.330912,"Kakinada to Mumbai/Pune Volvo AC Sleeper ₹2,50..."


In [34]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,12,passenger_conduct_and_safety_protocols,0.992625,"7. Luggage, Pets & Special Assistance 7.1 Lugg..."
1,8,booking_rules_and_ticketing_guidelines,1.145572,Defense Personnel: A 15% discount is provided ...
2,9,cancellation_rules_and_slabs,1.151215,4. Booking Rules & Ticketing Guidelines 4.1 Ti...
3,10,refund_rules_and_processing,1.154406,5. Cancellation Rules & Slabs 5.1 Standard Can...
4,7,booking_fares_and_pricing_structure,1.155828,3. Booking Fares & Pricing Structure 3.1 Dynam...
5,7,booking_fares_and_pricing_structure,1.156771,"Kakinada to Mumbai/Pune Volvo AC Sleeper ₹2,50..."


In [35]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,12,passenger_conduct_and_safety_protocols,0.507209,"7. Luggage, Pets & Special Assistance 7.1 Lugg..."
1,8,booking_rules_and_ticketing_guidelines,0.343765,Defense Personnel: A 15% discount is provided ...
2,9,cancellation_rules_and_slabs,0.337147,4. Booking Rules & Ticketing Guidelines 4.1 Ti...
3,10,refund_rules_and_processing,0.333322,5. Cancellation Rules & Slabs 5.1 Standard Can...
4,7,booking_fares_and_pricing_structure,0.331980,3. Booking Fares & Pricing Structure 3.1 Dynam...
5,7,booking_fares_and_pricing_structure,0.330898,"Kakinada to Mumbai/Pune Volvo AC Sleeper ₹2,50..."


In [36]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.50727951 0.34378799 0.3372162  0.3334392  0.33199665 0.33091197]

Dot product after normalization:
[0.50727951 0.34378799 0.3372162  0.3334392  0.33199665 0.33091197]

Are they approximately equal? True


In [39]:
cancellation_rules_and_slabs = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "cancellation_rules_and_slabs"
        },
    }
)

In [40]:
query = "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"

cancellation_rules_and_slabs = cancellation_rules_and_slabs.invoke(query)

display_documents(cancellation_rules_and_slabs)

RANK: 1
PAPER PAGE: 9
SECTION: cancellation_rules_and_slabs
CHUNK ID: sudheer-travels-9-chunk-11
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travels.pdf
------------------------------------------------------------------------------------------
4. Booking Rules & Ticketing Guidelines
4.1 Ticket Reservation and Validity
Advance Booking Window: Tickets can be booked up to 45 days in advance from
the date of the journey.
Ticketing Channels: Official website, mobile app, authorized agents, and offline
counters.
E-Tickets (m-Tickets): An SMS and Email containing the PNR serve as a valid e-
ticket.
Ticket Transferability: Tickets are strictly non-transferable. Name changes are not
permitted once confirmed.
4.2 Mandatory Identification at Boarding
Valid ID Proof is Mandatory: Passengers must present the original copy (or a valid digital
copy via DigiLocker) of a Government-issued Photo ID at the time of boarding. Failure to
produce valid 

RANK: 2


In [41]:
for document in cancellation_rules_and_slabs:
    assert document.metadata["section"] == "cancellation_rules_and_slabs"

print("All returned documents are from the cancellation_rules_and_slabs section.")

All returned documents are from the cancellation_rules_and_slabs section.


In [42]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        "$eq": "cancellation_rules_and_slabs"
                    }
                },
                {
                    "year": {
                        "$eq": 2026
                    }
                },
                {
                    "organization": {
                        "$eq": "Sudheer Travels Pvt. Ltd."
                    }
                },
            ]
        },
    }
)

In [43]:
query = "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 9
SECTION: cancellation_rules_and_slabs
CHUNK ID: sudheer-travels-9-chunk-11
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travels.pdf
------------------------------------------------------------------------------------------
4. Booking Rules & Ticketing Guidelines
4.1 Ticket Reservation and Validity
Advance Booking Window: Tickets can be booked up to 45 days in advance from
the date of the journey.
Ticketing Channels: Official website, mobile app, authorized agents, and offline
counters.
E-Tickets (m-Tickets): An SMS and Email containing the PNR serve as a valid e-
ticket.
Ticket Transferability: Tickets are strictly non-transferable. Name changes are not
permitted once confirmed.
4.2 Mandatory Identification at Boarding
Valid ID Proof is Mandatory: Passengers must present the original copy (or a valid digital
copy via DigiLocker) of a Government-issued Photo ID at the time of boarding. Failure to
produce valid 

RANK: 2


In [46]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "cancellation_rules_and_slabs"
            }
        },
        {
            "year": {
                "$eq": 2026
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 9
SECTION: cancellation_rules_and_slabs
CHUNK ID: sudheer-travels-9-chunk-11
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travels.pdf
------------------------------------------------------------------------------------------
4. Booking Rules & Ticketing Guidelines
4.1 Ticket Reservation and Validity
Advance Booking Window: Tickets can be booked up to 45 days in advance from
the date of the journey.
Ticketing Channels: Official website, mobile app, authorized agents, and offline
counters.
E-Tickets (m-Tickets): An SMS and Email containing the PNR serve as a valid e-
ticket.
Ticket Transferability: Tickets are strictly non-transferable. Name changes are not
permitted once confirmed.
4.2 Mandatory Identification at Boarding
Valid ID Proof is Mandatory: Passengers must present the original copy (or a valid digital
copy via DigiLocker) of a Government-issued Photo ID at the time of boarding. Failure to
produce valid 

RANK: 2


In [47]:
unfiltered_candidates = vector_store.similarity_search(
    query= "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?",
    k=15,
)

In [48]:
post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "cancellation_rules_and_slabs"
    and document.metadata.get("year") == 2026
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 9
SECTION: cancellation_rules_and_slabs
CHUNK ID: sudheer-travels-9-chunk-11
SOURCE: c:\Sudheer\Agentic_AI\workspace\python\AI_Basics\Basics_2\sudheer_travels\Sudheer_Travels.pdf
------------------------------------------------------------------------------------------
4. Booking Rules & Ticketing Guidelines
4.1 Ticket Reservation and Validity
Advance Booking Window: Tickets can be booked up to 45 days in advance from
the date of the journey.
Ticketing Channels: Official website, mobile app, authorized agents, and offline
counters.
E-Tickets (m-Tickets): An SMS and Email containing the PNR serve as a valid e-
ticket.
Ticket Transferability: Tickets are strictly non-transferable. Name changes are not
permitted once confirmed.
4.2 Mandatory Identification at Boarding
Valid ID Proof is Mandatory: Passengers must present the original copy (or a valid digital
copy via DigiLocker) of a Government-issued Photo ID at the time of boarding. Failure to
produce valid 

RANK: 2


In [49]:
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 2


In [51]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

In [52]:
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

In [55]:
bm25_retriever = BM25Retriever.from_documents(chunks)
print(bm25_retriever)
bm25_retriever.k = 4

vectorizer=<rank_bm25.BM25Okapi object at 0x000001FED7C9AA50>


In [56]:
sparse_query = "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
sparse_documents = bm25_retriever.invoke(sparse_query)

In [57]:
display_documents(
    sparse_documents,
    title="Sparse Retrieval: BM25 Results",
)


Sparse Retrieval: BM25 Results

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 8
Section: booking_rules_and_ticketing_guidelines
Chunk ID: sudheer-travels-8-chunk-10
------------------------------------------------

In [58]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

In [59]:
dense_query = (
    "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

dense_documents = dense_retriever.invoke(dense_query)

display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)


Dense Retrieval: Vector Search Results

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 8
Section: booking_rules_and_ticketing_guidelines
Chunk ID: sudheer-travels-8-chunk-10
----------------------------------------

In [60]:
comparison_query = (
    "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)


BM25 Results

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 8
Section: booking_rules_and_ticketing_guidelines
Chunk ID: sudheer-travels-8-chunk-10
------------------------------------------------------------------

In [61]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

SPARSE RESULTS
1 12 sudheer-travels-12-chunk-15
2 8 sudheer-travels-8-chunk-10
3 3 sudheer-travels-3-chunk-3
4 9 sudheer-travels-9-chunk-11

DENSE RESULTS
1 12 sudheer-travels-12-chunk-15
2 8 sudheer-travels-8-chunk-10
3 9 sudheer-travels-9-chunk-11
4 10 sudheer-travels-10-chunk-13


In [62]:
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 8
    },
)

In [63]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever,
    ],
    weights=[
        0.5,  # BM25 weight
        0.5,  # Dense-retrieval weight
    ],
)

In [65]:
hybrid_query = (
     "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

In [66]:
display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense + RRF",
    max_documents=6,
)


Hybrid Retrieval: BM25 + Dense + RRF

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 8
Section: booking_rules_and_ticketing_guidelines
Chunk ID: sudheer-travels-8-chunk-10
------------------------------------------

In [67]:
test_query = (
     "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

In [68]:
sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

In [69]:
display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)


1. Sparse Retrieval

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 8
Section: booking_rules_and_ticketing_guidelines
Chunk ID: sudheer-travels-8-chunk-10
-----------------------------------------------------------

In [70]:
CHAT_MODEL = os.environ.get(
    "OPENAI_CHAT_MODEL",
    "gpt-4.1-mini",
)

llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)

Chat model: gpt-4.1-mini


In [71]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.
""",
        ),
        (
            "human",
            """
Conversation history:
{chat_history}

Current query:
{query}
""",
        ),
    ]
)

In [72]:
query_rewriting_chain = (query_rewriting_prompt| llm | StrOutputParser())

In [73]:
chat_history = """
User:  "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
Assistant: It first underwent supervised fine-tuning.
"""

In [74]:
original_query = "How do I pay for it?"

In [75]:
rewritten_query = query_rewriting_chain.invoke(
    {
        "chat_history": chat_history,
        "query": original_query,
    }
).strip()

In [76]:
print("Original query:")
print(original_query)

print("\nRewritten query:")
print(rewritten_query)

Original query:
How do I pay for it?

Rewritten query:
How do I pay for extra luggage fees?


In [77]:
rewritten_query_documents = hybrid_retriever.invoke(
    rewritten_query
)

In [78]:
display_documents(
    rewritten_query_documents,
    title="Documents Retrieved Using the Rewritten Query",
    max_documents=5,
)


Documents Retrieved Using the Rewritten Query

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 8
Section: booking_rules_and_ticketing_guidelines
Chunk ID: sudheer-travels-8-chunk-10
---------------------------------

In [79]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

In [80]:
query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

In [81]:
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

In [82]:
query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

In [83]:
original_query = (
    "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

In [84]:
expanded_output = query_expansion_chain.invoke(
    {
        "query": original_query
    }
)

In [85]:
expanded_output

ExpandedQueryOutput(queries=['What is the maximum free baggage allowance per passenger and the fees for additional luggage?', 'How much free luggage weight can each passenger carry and what are the charges for excess baggage?', 'Maximum complimentary luggage weight per traveler and cost for extra baggage?', 'Allowed free baggage weight per passenger and the price for overweight luggage?'])

In [86]:
all_expanded_queries = [
    original_query,
    *expanded_output.queries,
]

In [87]:
all_expanded_queries

['What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?',
 'What is the maximum free baggage allowance per passenger and the fees for additional luggage?',
 'How much free luggage weight can each passenger carry and what are the charges for excess baggage?',
 'Maximum complimentary luggage weight per traveler and cost for extra baggage?',
 'Allowed free baggage weight per passenger and the price for overweight luggage?']

In [88]:
print("Generated search queries:\n")

for number, query in enumerate(
    all_expanded_queries,
    start=1,
):
    print(f"{number}. {query}")

Generated search queries:

1. What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?
2. What is the maximum free baggage allowance per passenger and the fees for additional luggage?
3. How much free luggage weight can each passenger carry and what are the charges for excess baggage?
4. Maximum complimentary luggage weight per traveler and cost for extra baggage?
5. Allowed free baggage weight per passenger and the price for overweight luggage?


In [89]:
expanded_query_documents = []

for query in all_expanded_queries:
    current_documents = dense_retriever.invoke(query)
    expanded_query_documents.extend(current_documents)

expanded_query_documents = deduplicate_documents(
    expanded_query_documents
)

display_documents(
    expanded_query_documents,
    title="Query Expansion: Combined Unique Documents",
    max_documents=10,
)


Query Expansion: Combined Unique Documents

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 8
Section: booking_rules_and_ticketing_guidelines
Chunk ID: sudheer-travels-8-chunk-10
------------------------------------

In [91]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm,
    include_original=True,
)

In [92]:
multi_query_documents = multi_query_retriever.invoke(
    "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

In [93]:
display_documents(
    multi_query_documents,
    title="Built-in MultiQueryRetriever Results",
    max_documents=10,
)


Built-in MultiQueryRetriever Results

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 2
Section: table_of_contents
Chunk ID: sudheer-travels-2-chunk-1
----------------------------------------------------------------

In [94]:
class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description=(
            "Independent and atomic search queries required "
            "to answer the complete user question."
        )
    )

In [95]:
query_decomposition_llm = llm.with_structured_output(
    DecomposedQueryOutput
)

In [96]:
query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Break the user's complex question into independent,
atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.
""",
        ),
        (
            "human",
            "Complex query: {query}",
        ),
    ]
)

In [97]:
query_decomposition_chain = (
    query_decomposition_prompt
    | query_decomposition_llm
)

In [98]:
complex_query = """
"Can I carry 25 kg of luggage along with a small pet inside a carrier on my trip, 
and how much will I be charged for the extra weight?"
"""

In [99]:
decomposed_output = query_decomposition_chain.invoke(
    {
        "query": complex_query
    }
)

In [100]:
sub_queries = decomposed_output.sub_queries

In [101]:
print("Original complex query:")
print(complex_query)

print("\nGenerated sub-queries:")

for number, query in enumerate(sub_queries, start=1):
    print(f"{number}. {query}")

Original complex query:

"Can I carry 25 kg of luggage along with a small pet inside a carrier on my trip, 
and how much will I be charged for the extra weight?"


Generated sub-queries:
1. What are the airline's luggage weight limits for passengers?
2. What are the airline's policies regarding carrying small pets inside carriers on trips?
3. What are the fees for excess luggage weight beyond the allowed limit?
4. Are there additional charges for carrying a small pet in a carrier during a trip?


In [102]:
decomposition_results = {}

for sub_query in sub_queries:
    decomposition_results[sub_query] = hybrid_retriever.invoke(
        sub_query
    )

In [103]:
for sub_query, documents in decomposition_results.items():
    display_documents(
        documents,
        title=f"Sub-query: {sub_query}",
        max_documents=4,
    )


Sub-query: What are the airline's luggage weight limits for passengers?

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 13
Section: customer_support_and_contact_information
Chunk ID: sudheer-travels-13-chunk-16
---

In [104]:
all_decomposition_documents = []

for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents)

all_decomposition_documents = deduplicate_documents(
    all_decomposition_documents
)

display_documents(
    all_decomposition_documents,
    title="Combined Evidence from All Sub-Queries",
    max_documents=12,
)


Combined Evidence from All Sub-Queries

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 13
Section: customer_support_and_contact_information
Chunk ID: sudheer-travels-13-chunk-16
------------------------------------

In [105]:
from typing import Any

from langchain_core.retrievers import BaseRetriever


class HyDERetriever(BaseRetriever):
    """Generate a hypothetical passage and retrieve real documents with it."""

    hypothesis_chain: Any
    vector_retriever: Any
    last_hypothetical_document: str = ""

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager,
    ):
        hypothetical_document = self.hypothesis_chain.invoke(
            {"query": query}
        ).strip()

        self.last_hypothetical_document = hypothetical_document

        return self.vector_retriever.invoke(
            hypothetical_document,
            config={
                "callbacks": run_manager.get_child(),
            },
        )

In [106]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Write a concise research-paper passage that would directly answer the user's question.
Use technical terminology and named entities likely to appear in the source document.
Do not mention that the passage is hypothetical.
Do not add citations, headings, or commentary.
Return only the passage.
""",
        ),
        ("human", "Question: {query}"),
    ]
)

hyde_hypothesis_chain = (
    hyde_prompt
    | llm
    | StrOutputParser()
)

hyde_dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

hyde_retriever = HyDERetriever(
    hypothesis_chain=hyde_hypothesis_chain,
    vector_retriever=hyde_dense_retriever,
)

In [107]:
hyde_query = (
  "The AC stopped working in the middle of our journey and never turned back on. Can I claim compensation?"
)

hyde_documents = hyde_retriever.invoke(hyde_query)

print("Original query:")
print(hyde_query)

print("\nHypothetical document used for retrieval:")
print(hyde_retriever.last_hypothetical_document)

display_documents(
    hyde_documents,
    title="HyDE Retrieval Results",
    max_documents=5,
)

Original query:
The AC stopped working in the middle of our journey and never turned back on. Can I claim compensation?

Hypothetical document used for retrieval:
If the air conditioning (AC) system in a vehicle ceased functioning during a journey and did not resume operation, eligibility for compensation depends on the terms of the transportation contract and applicable consumer protection laws. For commercial passenger transport, such as airlines, buses, or trains, failure to provide essential amenities like AC may constitute a breach of service standards, potentially entitling passengers to compensation or refunds under regulations such as the EU Regulation 261/2004 or the U.S. Department of Transportation guidelines. In private vehicle rentals, compensation claims typically require demonstrating that the malfunction significantly impaired the vehicle’s usability and that the rental company was notified and failed to remedy the issue. Documentation of the incident, communication wit

In [108]:
try:
    # LangChain v1: legacy chains live in langchain-classic.
    from langchain_classic.chains.hyde.base import (
        HypotheticalDocumentEmbedder,
    )
except ImportError:
    # Compatibility fallback for older LangChain versions.
    from langchain.chains.hyde.base import (
        HypotheticalDocumentEmbedder,
    )

In [111]:
built_in_hyde_embeddings = (
    HypotheticalDocumentEmbedder.from_llm(
        llm=llm,
        base_embeddings=embeddings,
        prompt_key="web_search",
    )
)

# Reuse the existing collection. Real PDF chunks remain embedded with
# `embeddings`; HyDE is used only to transform query embeddings.
built_in_hyde_vector_store = Chroma(
    collection_name="Sudheer_Travels_retriever_demo",
    embedding_function=built_in_hyde_embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

built_in_hyde_retriever = (
    built_in_hyde_vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5},
    )
)

In [112]:
built_in_hyde_query = (
    "The AC stopped working in the middle of our journey and never turned back on. Can I claim compensation?"
)

built_in_hyde_documents = (
    built_in_hyde_retriever.invoke(
        built_in_hyde_query
    )
)

display_documents(
    built_in_hyde_documents,
    title="Built-in LangChain HyDE Results",
    max_documents=5,
)


Built-in LangChain HyDE Results

RANK: 1
Paper page: 6
Section: booking_fares_and_pricing_structure
Chunk ID: sudheer-travels-6-chunk-7
----------------------------------------------------------------------------------------------------
every trip. Passengers are advised to carry personal hygiene items. In case of AC failure
during the journey, a partial refund policy applies.

RANK: 2
Paper page: 11
Section: luggage_pets_and_special_assistance
Chunk ID: sudheer-travels-11-chunk-14
----------------------------------------------------------------------------------------------------
6. Refund Rules & Processing
6.1 Refund Timelines and Methods
Processing Time: Initiated within 24 hours of request.
Bank Crediting Time: Reflects in source account within 5 to 7 business days.
Source Account Rule: Refunds ONLY credited to the original payment method.
Sudheer Wallet: Instant refund option valid for 365 days.
6.2 Refunds for Service Disruptions
Operator-Initiated Cancellations: 100% full refu

In [113]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 20
    },
)

In [114]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
)

In [115]:
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

In [116]:
reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [117]:
reranking_query = (
    "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

In [118]:
initial_candidates = candidate_retriever.invoke(
    reranking_query
)

In [119]:
initial_candidates

[Document(id='e221f9bb-871a-41d3-83d9-0dce47422b1e', metadata={'document_title': 'Sudheer Travels Official Rules & Regulations', 'total_pages': 14, 'creationdate': '2026-08-17T05:58:54+00:00', 'access_level': 'public_customer_facing', 'paper_page': 12, 'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'chunk_id': 'sudheer-travels-12-chunk-15', 'year': 2026, 'page_label': '12', 'document_type': 'customer_policies_and_guidelines', 'organization': 'Sudheer Travels Pvt. Ltd.', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\sudheer_travels\\Sudheer_Travels.pdf', 'page': 11, 'start_index': 0, 'section': 'passenger_conduct_and_safety_protocols', 'moddate': '2026-08-17T05:58:54+00:00'}, page_content='7. Luggage, Pets & Special Assistance\n7.1 Luggage Policy\nFree Allowance: 15 kg stowed luggage + one small cabin bag per passenger.\nExcess Luggage: Charged at ₹20 per kg, subject to availability.\nProhibited Items: Flammables, explosives, corrosives, contraband, wea

In [120]:
display_documents(
    initial_candidates,
    title="Before Reranking: Initial Vector Candidates",
    max_documents=10,
)


Before Reranking: Initial Vector Candidates

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 8
Section: booking_rules_and_ticketing_guidelines
Chunk ID: sudheer-travels-8-chunk-10
-----------------------------------

In [121]:
reranked_documents = reranking_retriever.invoke(reranking_query)

In [122]:
display_documents(
    reranked_documents,
    title="After Reranking: Final Top Documents",
    max_documents=5,
)


After Reranking: Final Top Documents

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 7
Section: booking_fares_and_pricing_structure
Chunk ID: sudheer-travels-7-chunk-9
----------------------------------------------

In [123]:
bm25_retriever.k = 15

In [124]:
dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 15
    },
)

In [125]:
hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_candidate_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

In [126]:
hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [127]:
query = (
    "What is the maximum free luggage weight allowed per passenger, and how much does extra luggage cost?"
)

In [128]:
final_documents = hybrid_reranking_retriever.invoke(query)

In [129]:
final_documents

[Document(metadata={'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'creationdate': '2026-08-17T05:58:54+00:00', 'moddate': '2026-08-17T05:58:54+00:00', 'source': 'c:\\Sudheer\\Agentic_AI\\workspace\\python\\AI_Basics\\Basics_2\\sudheer_travels\\Sudheer_Travels.pdf', 'total_pages': 14, 'page': 11, 'page_label': '12', 'document_title': 'Sudheer Travels Official Rules & Regulations', 'organization': 'Sudheer Travels Pvt. Ltd.', 'year': 2026, 'document_type': 'customer_policies_and_guidelines', 'paper_page': 12, 'section': 'passenger_conduct_and_safety_protocols', 'access_level': 'public_customer_facing', 'start_index': 0, 'chunk_id': 'sudheer-travels-12-chunk-15'}, page_content='7. Luggage, Pets & Special Assistance\n7.1 Luggage Policy\nFree Allowance: 15 kg stowed luggage + one small cabin bag per passenger.\nExcess Luggage: Charged at ₹20 per kg, subject to availability.\nProhibited Items: Flammables, explosives, corrosives, contraband, weapons,\nstrong-smelling goods.\nValuables: S

In [130]:
display_documents(
    final_documents,
    title="Hybrid Retrieval + Cross-Encoder Reranking",
    max_documents=5,
)


Hybrid Retrieval + Cross-Encoder Reranking

RANK: 1
Paper page: 12
Section: passenger_conduct_and_safety_protocols
Chunk ID: sudheer-travels-12-chunk-15
----------------------------------------------------------------------------------------------------
7. Luggage, Pets & Special Assistance
7.1 Luggage Policy
Free Allowance: 15 kg stowed luggage + one small cabin bag per passenger.
Excess Luggage: Charged at ₹20 per kg, subject to availability.
Prohibited Items: Flammables, explosives, corrosives, contraband, weapons,
strong-smelling goods.
Valuables: Sudheer Travels assumes no liability for lost or stolen valuables.
7.2 Pet Policy
No Pets Allowed: Pets are strictly prohibited inside the passenger cabin and luggage
compartments.
7.3 Passengers Requiring Special Assistance
Wheelchair Assistance: Ground staff can assist. Notify 24 hours in adva

RANK: 2
Paper page: 7
Section: booking_fares_and_pricing_structure
Chunk ID: sudheer-travels-7-chunk-9
----------------------------------------